# Build WGMS–RGI targets and the ERA5-Land inputs

Read the input paths and Earth Engine export instructions in the release README. Run sections 1–2 before exporting forcing for a new run, then resume at section 4 after downloading the exports.

## 1. Build the WGMS-RGI annual target table

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

from os import environ

configured_root = environ.get("WGMS_MB_ROOT")
ROOT_DIR = Path(configured_root).expanduser().resolve() if configured_root else Path.cwd().resolve()
if not configured_root:
    if (ROOT_DIR / "01_Dataset.ipynb").is_file():
        ROOT_DIR = ROOT_DIR.parent
    if not (ROOT_DIR / "code" / "01_Dataset.ipynb").is_file():
        raise RuntimeError("Start this notebook from the release root or its code directory, or set WGMS_MB_ROOT.")
elif not ROOT_DIR.is_dir():
    raise FileNotFoundError(f"The configured WGMS_MB_ROOT directory does not exist: {ROOT_DIR}")
WGMS_DIR = ROOT_DIR / "data/DOI-WGMS-FoG-2026-02-10/data"
RGI6_DIR = ROOT_DIR / "data/RGI6"
OUT_DIR = ROOT_DIR / "data/interim/wgms_targets"
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_MIN, YEAR_MAX = 1979, 2025
FORCING_YEAR_MIN = 1978
RGI_STATIC_COLS = [
    'GLIMSId','BgnDate','EndDate','CenLon','CenLat','O1Region','O2Region',
    'Area','Zmin','Zmax','Zmed','Slope','Aspect','Lmax',
    'Status','Connect','Form','TermType','Surging','Linkages','Name'
]

print(f"WGMS_DIR = {WGMS_DIR}")
print(f"RGI6_DIR = {RGI6_DIR}")
print(f"OUT_DIR = {OUT_DIR}")
print(f"YEAR_RANGE = {YEAR_MIN}-{YEAR_MAX}")

In [ ]:
glacier = pd.read_csv(WGMS_DIR / 'glacier.csv', low_memory=False)
mass_balance = pd.read_csv(WGMS_DIR / 'mass_balance.csv', low_memory=False)
print(glacier.shape, mass_balance.shape)
display(glacier.head())
display(mass_balance.head())

In [ ]:
wgms_mb = mass_balance[mass_balance['annual_balance'].notna()].copy()
wgms_mb = wgms_mb[wgms_mb['year'].between(YEAR_MIN, YEAR_MAX)].copy()
wgms_join = wgms_mb.merge(glacier, left_on='glacier_id', right_on='id', how='inner', validate='many_to_one')
wgms_join['rgiid'] = wgms_join['rgi60_ids'].fillna('').astype(str).str.split(' | ').str[0]
wgms_join['has_rgiid'] = wgms_join['rgiid'].ne('')

print('WGMS glaciers with annual balance =', wgms_join['glacier_id'].nunique())
print('Annual-balance rows =', len(wgms_join))
print('Mapped to RGI6 =', int(wgms_join['has_rgiid'].sum()), 'rows')
display(wgms_join[['glacier_id','short_name','year','annual_balance','rgi60_ids','rgiid','has_rgiid']].head())

In [ ]:
wgms_audit = (
    wgms_join.groupby('glacier_id', as_index=False)
    .agg(
        short_name=('short_name', 'first'),
        gtng_region=('gtng_region', 'first'),
        years=('year', 'nunique'),
        first_year=('year', 'min'),
        last_year=('year', 'max'),
        has_rgiid=('has_rgiid', 'max'),
        rgiid=('rgiid', 'first'),
        rgi70_ids=('rgi70_ids', 'first'),
        glims_id=('glims_id', 'first'),
        latitude=('latitude', 'first'),
        longitude=('longitude', 'first'),
        time_system_sample=('time_system', 'first')
    )
    .sort_values(['has_rgiid', 'gtng_region', 'glacier_id'], ascending=[True, True, True])
)

wgms_audit.to_csv(OUT_DIR / 'wgms_global_glacier_audit.csv', index=False)
wgms_audit.loc[~wgms_audit['has_rgiid']].to_csv(OUT_DIR / 'wgms_global_missing_rgi_mapping.csv', index=False)
print('Missing RGI mapping glaciers =', int((~wgms_audit['has_rgiid']).sum()))
display(wgms_audit.head(20))

In [ ]:
rgi_paths = sorted([
    p for p in RGI6_DIR.glob('*_rgi60_*.csv')
    if '_hypso' not in p.name and p.name[:2].isdigit() and p.name[:2] != '00' and '_summary' not in p.name and '_links' not in p.name
])
rgi_frames = [pd.read_csv(p, engine='python', encoding='latin1', on_bad_lines='skip') for p in rgi_paths]
glacier_meta = pd.concat(rgi_frames, ignore_index=True).drop_duplicates('RGIId')
glacier_meta = glacier_meta[['RGIId'] + RGI_STATIC_COLS].copy()

print('Loaded RGI region tables =', len(rgi_paths))
print(glacier_meta.shape)
display(glacier_meta[['RGIId','Area','O1Region','O2Region']].head())

In [ ]:
wgms_target = (
    wgms_join[wgms_join['has_rgiid']].copy()
    .groupby(['rgiid', 'year'], as_index=False)
    .agg(
        mwe=('annual_balance', 'mean'),
        wgms_area_m2=('area', 'mean'),
        time_system=('time_system', 'first'),
        glacier_name=('short_name', 'first'),
        gtng_region=('gtng_region', 'first'),
        wgms_glacier_id=('glacier_id', 'first'),
        n_records=('annual_balance', 'size')
    )
    .rename(columns={'rgiid': 'RGIId'})
)

merged = wgms_target.merge(glacier_meta, on='RGIId', how='inner')
merged['Area_rgi'] = merged['Area']
merged['wgms_area_km2'] = merged['wgms_area_m2'] / 1e6
merged['Area_source'] = 'RGI'
invalid_area = merged['Area'].isna() | ~np.isfinite(merged['Area']) | merged['Area'].le(0)
if invalid_area.any():
    raise ValueError(f"Missing or invalid RGI 6.0 area for {merged.loc[invalid_area, 'RGIId'].unique()[:10].tolist()}")

out_csv = OUT_DIR / 'wgms_global_targets_with_rgi_1979_2025.csv'
merged.to_csv(out_csv, index=False)

summary = {
    'dataset_path': out_csv.relative_to(ROOT_DIR).as_posix(),
    'rows': int(len(merged)),
    'glaciers': int(merged['RGIId'].nunique()),
    'year_min': int(merged['year'].min()),
    'year_max': int(merged['year'].max()),
    'mapped_wgms_glaciers': int(wgms_audit['has_rgiid'].sum()),
    'missing_rgi_glaciers': int((~wgms_audit['has_rgiid']).sum()),
    'o1_regions': sorted(pd.Series(merged['O1Region']).dropna().astype(int).unique().tolist()),
    'wgms_area_rows': int(merged['wgms_area_km2'].notna().sum()),
    'area_source_counts': merged['Area_source'].value_counts(dropna=False).to_dict()
}
audit = {
    'wgms_glaciers_with_annual_balance': int(wgms_join['glacier_id'].nunique()),
    'wgms_annual_balance_rows': int(len(wgms_join)),
    'wgms_glaciers_with_rgi60_id': int(wgms_audit['has_rgiid'].sum()),
    'wgms_glaciers_missing_rgi60_id': int((~wgms_audit['has_rgiid']).sum()),
    'time_system_counts': wgms_join['time_system'].fillna('NA').value_counts().to_dict()
}
(OUT_DIR / 'wgms_global_dataset_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
(OUT_DIR / 'wgms_global_target_audit.json').write_text(json.dumps(audit, indent=2, ensure_ascii=False), encoding='utf-8')

print(summary)
print(audit)
display(merged.head())

## 2. Prepare the Earth Engine upload table

In [ ]:
from pathlib import Path
import pandas as pd


IN_CSV = ROOT_DIR / "data/interim/wgms_targets/wgms_global_targets_with_rgi_1979_2025.csv"
OUT_DIR = ROOT_DIR / "data/raw/era5land_gee"
OUT_DIR.mkdir(parents=True, exist_ok=True)
POINT_CSV = OUT_DIR / "wgms_global_glacier_points_for_gee.csv"
SAMPLE_CSV = OUT_DIR / "wgms_global_glacier_years_for_gee.csv"
GEE_SCRIPT = ROOT_DIR / "code/gee_export_era5land.js"

print(f"Input target table: {IN_CSV}")
print(f"GEE point upload CSV: {POINT_CSV}")
print(f"GEE Code Editor script: {GEE_SCRIPT}")

In [ ]:
data = pd.read_csv(IN_CSV, low_memory=False)
point_df = (
    data[['RGIId','CenLon','CenLat','O1Region','O2Region']]
    .dropna(subset=['RGIId','CenLon','CenLat'])
    .drop_duplicates('RGIId')
    .reset_index(drop=True)
)
sample_df = data[['RGIId','year']].drop_duplicates().sort_values(['year','RGIId']).reset_index(drop=True)

point_df.to_csv(POINT_CSV, index=False)
sample_df.to_csv(SAMPLE_CSV, index=False)

print(data.shape)
print('Unique glaciers for GEE sampling =', len(point_df))
print('Unique glacier-year target rows =', len(sample_df))
print(f'Wrote point CSV to: {POINT_CSV}')
print(f'Wrote glacier-year CSV to: {SAMPLE_CSV}')
display(point_df.head())

## 3. Export ERA5-Land in the Earth Engine Code Editor

Upload the generated centroid CSV to Earth Engine, set `POINT_ASSET` in `code/gee_export_era5land.js`, and run the export task. Download all exported CSV files to `data/raw/era5land_gee/exports/`. If the already merged project export is available, it can be placed at `data/raw/era5land_gee/wgms_global_era5land_1979_2025.csv` instead. The data were sampled at 11,132 m from the ERA5-Land `MONTHLY_AGGR` collection for the 1979–2025 balance years. An optional 1978 export can be added in `data/raw/era5land_gee/extension/` to complete cross-year seasonal windows.

## 4. Merge downloaded ERA5-Land exports

In [ ]:
from pathlib import Path
import json
import pandas as pd


BASE_CSV = ROOT_DIR / "data/interim/wgms_targets/wgms_global_targets_with_rgi_1979_2025.csv"
EXPORT_DIR = ROOT_DIR / "data/raw/era5land_gee/exports"
EXTENSION_DIR = EXPORT_DIR.parent / "extension"
ERA_LONG_CSV = EXPORT_DIR.parent / "wgms_global_era5land_1979_2025.csv"
RGI_REGION_SUPPORT = ROOT_DIR / "models/uncertainty/rgi_region_support_for_uncertainty.csv"
OUT_DIR = ROOT_DIR / "data/interim/wgms_era5land_merged"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GEE_TO_SHORT = {
    'temperature_2m': 't2m',
    'snow_density': 'rsn',
    'snow_depth_water_equivalent': 'sde',
    'snowfall_sum': 'sf',
    'snowmelt_sum': 'smlt',
    'temperature_of_snow_layer': 'tsn',
    'forecast_albedo': 'fal',
    'surface_latent_heat_flux_sum': 'slhf',
    'surface_net_solar_radiation_sum': 'ssr',
    'surface_thermal_radiation_downwards_sum': 'strd',
    'surface_sensible_heat_flux_sum': 'sshf',
    'surface_solar_radiation_downwards_sum': 'ssrd',
    'surface_pressure': 'sp',
    'total_precipitation_sum': 'tp'
}

print(f"BASE_CSV = {BASE_CSV}")
print(f"EXPORT_DIR = {EXPORT_DIR}")

print(f"OUT_DIR = {OUT_DIR}")

In [ ]:
base = pd.read_csv(BASE_CSV, low_memory=False)
export_paths = sorted(EXPORT_DIR.glob('*.csv'))
if not export_paths and not ERA_LONG_CSV.is_file():
    raise FileNotFoundError(
        f"Download the Earth Engine CSVs to {EXPORT_DIR}, or place the merged export at {ERA_LONG_CSV}."
    )

keys = ['RGIId', 'year', 'month']
variables = list(GEE_TO_SHORT.values())
short_cols = keys + variables
long_cols = keys + list(GEE_TO_SHORT)


def read_export(path):
    # Retain the original decimal tokens when writing the normalized export.
    table = pd.read_csv(path, dtype='string', low_memory=False)
    if set(short_cols).issubset(table.columns):
        table = table[short_cols].copy()
    elif set(long_cols).issubset(table.columns):
        table = table[long_cols].rename(columns=GEE_TO_SHORT)
    else:
        missing = sorted(set(short_cols).difference(table.columns))
        raise ValueError(f"Missing expected forcing columns in {path}: {missing}")
    table['RGIId'] = table['RGIId'].str.strip()
    if table['RGIId'].isna().any() or table['RGIId'].eq('').any():
        raise ValueError(f"Missing glacier identifiers in {path}")
    for key, lower, upper in [('year', FORCING_YEAR_MIN, YEAR_MAX), ('month', 1, 12)]:
        values = pd.to_numeric(table[key], errors='raise')
        in_range = values.between(lower, upper) & values.mod(1).eq(0)
        if not in_range.fillna(False).all():
            raise ValueError(f"Invalid {key} values in {path}")
        table[key] = values.astype(int)
    table[variables].apply(pd.to_numeric, errors='raise')
    return table


extension_paths = sorted(EXTENSION_DIR.glob('*.csv'))
input_paths = (export_paths or [ERA_LONG_CSV]) + extension_paths
export = pd.concat([read_export(path) for path in input_paths], ignore_index=True)
if export.empty:
    raise ValueError("The ERA5-Land input files contain no monthly records.")
source_row_count = len(export)
export = export.drop_duplicates()
repeated = export.duplicated(keys, keep=False)
if repeated.any():
    overlap = export.loc[repeated].copy()
    overlap[variables] = overlap[variables].apply(pd.to_numeric, errors='raise')
    overlap = overlap.drop_duplicates()
    conflicts = overlap.duplicated(keys, keep=False)
    if conflicts.any():
        examples = overlap.loc[conflicts, keys].drop_duplicates().head().to_dict('records')
        raise ValueError(f"Conflicting monthly values for the same glacier and month: {examples}")
    export = export.drop_duplicates(keys)

# Read every mode from the same canonical file via the default parser. An
# already normalized historical export can be reused without rewriting it.
headers = pd.read_csv(ERA_LONG_CSV, nrows=0).columns if ERA_LONG_CSV.is_file() else []
if export_paths or extension_paths or not set(short_cols).issubset(headers) or len(export) != source_row_count:
    ERA_LONG_CSV.parent.mkdir(parents=True, exist_ok=True)
    export.to_csv(ERA_LONG_CSV, index=False)
era_long = pd.read_csv(ERA_LONG_CSV, usecols=short_cols, low_memory=False)
print(f"Monthly forcing: {len(era_long):,} rows from {len(input_paths)} file(s)")
display(era_long.head())

In [ ]:
wide_frames = []
for month in range(1, 13):
    month_df = era_long[era_long['month'] == month].copy()
    month_df = month_df.rename(columns={src: f'{month}_{src}' for src in GEE_TO_SHORT.values()})
    month_keep = ['RGIId', 'year'] + [f'{month}_{dst}' for dst in GEE_TO_SHORT.values()]
    wide_frames.append(month_df[month_keep])

era_wide = wide_frames[0]
for frame in wide_frames[1:]:
    era_wide = era_wide.merge(frame, on=['RGIId', 'year'], how='outer', validate='one_to_one')

print(era_wide.shape)
display(era_wide.head())

In [ ]:
merged = base.merge(era_wide, on=['RGIId', 'year'], how='inner', validate='one_to_one')
out_csv = OUT_DIR / 'wgms_global_rgi_era5land_1979_2025.csv'
merged.to_csv(out_csv, index=False)

summary = {
    'dataset_path': out_csv.relative_to(ROOT_DIR).as_posix(),
    'rows': int(len(merged)),
    'glaciers': int(merged['RGIId'].nunique()),
    'year_min': int(merged['year'].min()),
    'year_max': int(merged['year'].max()),
    'o1_regions': sorted(pd.Series(merged['O1Region']).dropna().astype(int).unique().tolist()),
    'monthly_var_count': int(len(GEE_TO_SHORT) * 12)
}
(OUT_DIR / 'wgms_global_rgi_era5land_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print(summary)
display(merged.head())


# Derive publication aggregation weights from all RGI glacier attributes.
# Sample statistics use the glacier set retained after the ERA5-Land merge.
# Restrict the parse to the area lookup fields. The official Central Asia
# attributes contain a row with a trailing delimiter after its last field.
area_fields = ['RGIId', 'O1Region', 'Area']
rgi_areas = pd.concat([
    pd.read_csv(path, engine='python', encoding='latin1', usecols=area_fields)
    for path in rgi_paths
], ignore_index=True).drop_duplicates('RGIId')
rgi_areas['O1Region'] = pd.to_numeric(rgi_areas['O1Region'], errors='raise')
rgi_areas['Area'] = pd.to_numeric(rgi_areas['Area'], errors='raise')
region_weights = rgi_areas.groupby('O1Region').agg(rgi_n=('RGIId', 'size'), rgi_area=('Area', 'sum'))
if set(region_weights.index) != set(range(1, 20)) or not region_weights['rgi_area'].gt(0).all():
    raise ValueError('RGI 6.0 area weighting requires all 19 first-order region attribute files.')
sample = merged[['RGIId', 'O1Region', 'Area']].drop_duplicates('RGIId')
sample_weights = sample.groupby('O1Region').agg(sample_n=('RGIId', 'size'), sample_area=('Area', 'sum'))
region_weights = region_weights.join(sample_weights).fillna(0)
region_weights['sample_n'] = region_weights['sample_n'].astype(int)
region_weights['area_coverage'] = region_weights['sample_area'] / region_weights['rgi_area']
RGI_REGION_SUPPORT.parent.mkdir(parents=True, exist_ok=True)
region_weights.to_csv(RGI_REGION_SUPPORT)
print(f"RGI area weights: {len(region_weights)} first-order regions")